# Processamento dos slides

Primeiramente, a conversão é de PPTX para TXT. É possível pular essa etapa e ir diretamente para JSON, mas a conversão para TXT é útil para visualização e depuração.

In [ ]:
from glob import glob
from tqdm import tqdm
from pathlib import Path
from pptx import Presentation
from pptx.enum.shapes import MSO_SHAPE_TYPE
import unicodedata

files = glob(str(Path("slides_adapt") / "*.pptx"))
files

In [ ]:
file = files[-2]
prs = Presentation(file)
prs

In [ ]:
print(f"Slides count: {len(prs.slides)}")
print(f"Slide width: {prs.slide_width}")
print(f"Slide height: {prs.slide_height}")
print(f"\nCore properties:")
print(f"  Title: {prs.core_properties.title}")
print(f"  Author: {prs.core_properties.author}")
print(f"  Subject: {prs.core_properties.subject}")
print(f"  Created: {prs.core_properties.created}")
print(f"  Modified: {prs.core_properties.modified}")

# Detailed slide information
print("\n=== Detailed Slide Information ===\n")
for slide_idx, slide in enumerate(prs.slides):
    print(f"\n--- Slide {slide_idx} ---")
    print(f"Shapes count: {len(slide.shapes)}")
    
    for shape_idx, shape in enumerate(slide.shapes):
        print(f"\n  Shape {shape_idx}:")
        print(f"    All shape keys: {dir(shape)}")
        print(f"    Type: {shape.shape_type} ({shape.shape_type.name if hasattr(shape.shape_type, 'name') else 'N/A'})")
        print(f"    Name: {shape.name}")
        print(f"    Rotation: {shape.rotation}")
        print(f"    Left: {shape.left}, Top: {shape.top}")
        print(f"    Width: {shape.width}, Height: {shape.height}")
        
        if hasattr(shape, 'text') and shape.text:
            print(f"    Text: {repr(shape.text[:100])}{'...' if len(shape.text) > 100 else ''}")
        
        # if hasattr(shape, 'auto_shape_type'):
        if shape.shape_type == MSO_SHAPE_TYPE.AUTO_SHAPE:
            print(f"    Auto Shape Type: {shape.auto_shape_type}")


In [ ]:
for shape in prs.slides[23].shapes:
    print(f"SHAPE_{shape.shape_type}")

    if shape.shape_type == MSO_SHAPE_TYPE.AUTO_SHAPE:
        print(f"_AUTO_SHAPE_{shape.auto_shape_type}\n")
    elif shape.shape_type == MSO_SHAPE_TYPE.FREEFORM:
        print(f"_FREEFORM_\n")
        

    if hasattr(shape, "text"):
        print(f"\nSTART_TEXT\n{shape.text}\nEND_TEXT\n")

    print(f"HEIGHT_{shape.height}\n")
    print(f"TOP_{shape.top}\n")

    print("\n")

In [ ]:
import unicodedata
from typing import Mapping

PUNCT_MAP: Mapping[int, str] = {
    # Dashes / hyphens
    0x2010: "-",  # HYPHEN
    0x2011: "-",  # NON-BREAKING HYPHEN
    0x2012: "-",  # FIGURE DASH
    0x2013: "-",  # EN DASH
    0x2014: "-",  # EM DASH
    0x2015: "-",  # HORIZONTAL BAR
    # Quotes
    0x2018: "'",  # LEFT SINGLE QUOTATION MARK
    0x2019: "'",  # RIGHT SINGLE QUOTATION MARK
    0x201B: "'",  # SINGLE HIGH-REVERSED-9 QUOTATION MARK
    0x201C: '"',  # LEFT DOUBLE QUOTATION MARK
    0x201D: '"',  # RIGHT DOUBLE QUOTATION MARK
    0x201F: '"',  # DOUBLE HIGH-REVERSED-9 QUOTATION MARK
    # Misc punctuation / spacing
    0x00A0: " ",  # NO-BREAK SPACE
    0x2007: " ",  # FIGURE SPACE
    0x2009: " ",  # THIN SPACE
    0x2026: "...",  # HORIZONTAL ELLIPSIS
}


def normalize_text(s: str, ascii_only: bool = False) -> str:
    # Step 1: Unicode normalization (helps with compatibility characters)
    s = unicodedata.normalize("NFKC", s)

    # Step 2: map “fancy” punctuation to simple ASCII
    s = "".join(PUNCT_MAP.get(ord(ch), ch) for ch in s)

    if ascii_only:
        # Step 3: force pure ASCII, dropping the rest
        # (change to 'replace' if you prefer ? instead of dropping)
        s = s.encode("ascii", "ignore").decode("ascii")
    return s


for file in files:
    new_file = Path("slides_txt") / (Path(file).stem + ".txt")

    f = open(new_file, "w", encoding="utf-8")

    prs = Presentation(file)
    for i, slide in tqdm(enumerate(prs.slides), desc="Processing slides", total=len(prs.slides)):

        f.write(f"\nSLIDE_{i}\n")

        for shape in slide.shapes:
            f.write(f"SHAPE_{shape.shape_type}")

            if shape.shape_type == MSO_SHAPE_TYPE.AUTO_SHAPE:
                f.write(f"_AUTO_SHAPE_{shape.auto_shape_type}\n")
            elif shape.shape_type == MSO_SHAPE_TYPE.FREEFORM:
                f.write("_SHAPE_FREEFORM_")

            if hasattr(shape, "text"):
                if shape.text.strip() != "":
                    f.write(f"\nSTART_TEXT\n{normalize_text(shape.text)}\nEND_TEXT\n")

            f.write(f"\nHEIGHT_{shape.height}\n")
            f.write(f"TOP_{shape.top}\n")

            f.write("\n")

        for shape in slide.shapes:
            if hasattr(shape, "text"):
                if normalize_text(shape.text).strip().lower() in ["indice", "índice"]:
                    f.write("\n__END__\n")

    f.close()

# Processamento do texto

In [ ]:
import re
import json
from glob import glob
from pathlib import Path

files_txt = glob(str(Path("slides_txt") / "*.txt"))
print(f"Files found: {files_txt}")

In [ ]:
with open(files_txt[-2], "r", encoding="utf-8") as f:
    text = f.read()

print(text.split("__END__")[0])

In [ ]:
file_txt = files_txt[-2]
auto_shape_pattern = r"_AUTO_SHAPE_([A-Z_]+)"
freeform_pattern = r"_FREEFORM_"


print(f"Processing file: {file_txt}")

with open(file_txt, "r", encoding="utf-8") as f:
    text = f.read()

praise = text.split("__END__")[0]
praise_struc = {}
praise_struc["slides"] = []

for slide in praise.split("SLIDE_"):
    slide_struc = {}
    slide_num = slide.split("\n")[0].strip()
    if not slide_num:
        continue

    slide_struc["slide"] = slide_num
    slide_struc["shapes"] = []

    for shape in slide.split("\nSHAPE_"):
        record_line = False
        text_inside = ""
        shapes_struc = {}

        for line in shape.split("\n"):
            if "SHAPE_" in line:
                match_auto_shape = re.search(auto_shape_pattern, line)
                if match_auto_shape:
                    auto_shape = match_auto_shape.group(1)
                    shapes_struc["shape"] = "AUTO_SHAPE"
                    shapes_struc["auto_shape"] = auto_shape

                match_freeform = re.search(freeform_pattern, line)
                if match_freeform:
                    shapes_struc["shape"] = "FREEFORM"

            if "HEIGHT_" in line:
                height = line.split("_")[1]
                shapes_struc["height"] = int(height)
            if "TOP_" in line:
                top = line.split("_")[1]
                shapes_struc["top"] = int(top)

            if "END_TEXT" in line:
                record_line = False

            if record_line:
                text_inside += line + "\n"
            else:
                if text_inside:
                    shapes_struc["text"] = text_inside#.strip()

            if "START_TEXT" in line:
                text_inside = ""
                record_line = True

        if shapes_struc:
            if text_inside.strip().lower() not in ["índice"]:
                slide_struc["shapes"].append(shapes_struc)

    praise_struc["slides"].append(slide_struc)

print(json.dumps(praise_struc, ensure_ascii=False, indent=4))

# Processamento continuado do ~JSON~ texto

O plano:
1. Processar todos os "text" dos shapes, aplicando as tags necessárias.
2. Dentro de cada slide que tem um right brace ou um freeform, identificar quais shapes são de texto real (tem que ter \<bis\>)

In [ ]:
from pathlib import Path
import json
from glob import glob

# json_file = Path("slides_json/LOUVORES AVULSOS_Rev_31.12.22_ADAPT.json")
# json_file = Path("slides_json/01.COLETÂNEA_IGREJAS_2022_TV-16.9_ADAPT.json")
# json_file = Path("slides_json/03.COLETÂNEA DE CIAS_2021 TV_ADAPT.json")
# json_file = Path("slides_json/Louvores Avulsos CIAs 2022 - COM ANIMAÇÃO_ADAPT.json")

all_praises = []
json_files = glob(str(Path("slides_json") / "*.json"))
for json_file in json_files:
    with open(json_file, "r") as f:
        praises = json.load(f)
    all_praises.extend(praises)

# all_praises[0]

## Tags

In [ ]:
for praise in all_praises:
    for slide in praise["slides"]:
        for shape in slide["shapes"]:
            if "text" in shape:
                if "(3X)" in shape["text"]:
                    print("----")
                    print(shape["text"])
                    print("----")

In [ ]:
import re

coro_regex = r"\s*CORO(:)?\s*(\(BIS\)|\(?\d\s*X\)?)?(:)?\s*\n"  # o segundo grupo é a quantidade de repetições
instrumentos_regex = r"\n\s*INSTRUMENTOS\s*(\n|$)"
final_regex = r"(\n|^)\s*FINAL\s*(:)?\s*"
bis_regex = r"\(?BIS(\s+NO\s+FINAL|\s*\dX)?\)?\s*$"

varoes_regex = r"\(?VARÕES\)?\s*(\n|$)"
servas_regex = r"\(?SERVAS\)?"
h_regex = r"\(H\)|^\s*H\s*$"  # H sozinho em uma linha ou (H)
s_regex = r"\(S\)|^\s*S\s*$"  # S sozinho em uma linha ou (S)
m_regex = r"\(M\)|^\s*M\s*$"  # M sozinho em uma linha ou (M)
t_regex = r"\(T\)|^\s*T\s*$"  # T sozinho em uma linha ou (T)

repetir_regex = r"REPETIR\s+(A\s+1a\s+ESTROFE\s+\dX|O\s+LOUVOR|O\s+HINO)"
repetir_no_final_regex = r"\((\d+)X\s+NO\s+FINAL\)"
repetir_x_vezes_regex = r"\((\d+)X\)"


def _normalize_tag_value(value: str) -> str:
    value = re.sub(r"[()]+", " ", value)
    value = re.sub(r"\s+", " ", value).strip().upper()
    return value

def tagify_text(text: str) -> str:
    if not text:
        return text

    tagged = text

    # \t e \n
    tagged = tagged.replace("\t\n", "\n")
    tagged = tagged.replace("\t", "\n")

    # CORO (com repetição opcional)
    def repl_coro(match: re.Match) -> str:
        rep = match.group(2)
        if rep:
            rep = _normalize_tag_value(rep)
            return f"<coro tipo=\"{rep}\">\n"
        return "<coro>\n"

    tagged = re.sub(
        coro_regex,
        repl_coro,
        tagged,
        flags=re.IGNORECASE | re.MULTILINE,
    )

    # Repetir
    def repl_repetir(match: re.Match) -> str:
        tipo = match.group(1)
        if tipo:
            tipo = _normalize_tag_value(tipo)
            return f"<repetir tipo=\"{tipo}x\">"
        return "<repetir>"

    tagged = re.sub(
        repetir_regex,
        repl_repetir,
        tagged,
        flags=re.IGNORECASE,
    )

    # Instrumentos / Final
    tagged = re.sub(
        instrumentos_regex,
        "\n<instrumentos>\n",
        tagged,
        flags=re.IGNORECASE,
    )
    tagged = re.sub(
        final_regex,
        "\n<final>",
        tagged,
        flags=re.IGNORECASE,
    )

    # BIS (incluindo casos como BIS NO FINAL ou BIS 2X)
    def repl_bis(match: re.Match) -> str:
        modo = match.group(1)
        if modo:
            modo = _normalize_tag_value(modo)
            return f"<bis tipo=\"{modo}\">"
        return "<bis>"

    tagged = re.sub(
        bis_regex,
        repl_bis,
        tagged,
        flags=re.IGNORECASE | re.MULTILINE,
    )

    # Repetir X vezes (considerado como bis com modo)
    def repl_repetir_x_vezes(match: re.Match) -> str:
        vezes = match.group(1)
        return f'<bis modo="{vezes}">'
    
    tagged = re.sub(
        repetir_x_vezes_regex,
        repl_repetir_x_vezes,
        tagged,
        flags=re.IGNORECASE,
    )

    # Vozes
    tagged = re.sub(
        varoes_regex,
        "<h>\n",
        tagged,
        flags=re.IGNORECASE,
    )
    tagged = re.sub(
        servas_regex,
        "<m>",
        tagged,
        flags=re.IGNORECASE,
    )
    tagged = re.sub(h_regex, "<h>", tagged, flags=re.IGNORECASE)
    tagged = re.sub(s_regex, "<m>", tagged, flags=re.IGNORECASE)
    tagged = re.sub(m_regex, "<m>", tagged, flags=re.IGNORECASE)
    tagged = re.sub(t_regex, "<t>", tagged, flags=re.IGNORECASE)
    
    # (2X NO FINAL) -> <repetir tipo="2">
    tagged = re.sub(
        repetir_no_final_regex,
        lambda m: f'<repetir tipo="no-final,{m.group(1)}x">',
        tagged,
        flags=re.IGNORECASE,
    )

    return tagged

def apply_tags_to_all_praises(praises: list):
    for praise in praises:
        for slide in praise.get("slides", []):
            for shape in slide.get("shapes", []):
                original_text = shape.get("text")
                if not original_text:
                    continue

                tagged_text = tagify_text(original_text)

                shape["text"] = original_text
                shape["text_processado"] = tagged_text


apply_tags_to_all_praises(all_praises)

In [ ]:
all_praises

## Identificação de título

In [ ]:
def get_texts(d, key="text_processado"):
    if isinstance(d, dict):
        for k, v in d.items():
            if k == key:
                yield v
            else:
                yield from get_texts(v, key=key)
    elif isinstance(d, list):
        for v in d:
            yield from get_texts(v, key=key)

In [ ]:
def return_possible_title(texts:list):
    possible_title = [
        text
        for text in texts
        if "<" not in text
        and text.strip() != ""
    ]
    if possible_title:
        min_string = min(possible_title, key=len)
        title = min_string.upper().replace("\n", " ").strip()
        
        return title, min_string
        
    return None, None


In [ ]:
# procurar slides com bis
for praise in all_praises:
    # o titulo deve estar no primeiro slide
    first_slide = praise["slides"][0]["shapes"]
    first_slide_texts = list(get_texts(first_slide))
    title, unchanged_title = return_possible_title(first_slide_texts)
    print(f"Title: {title}, Unchanged Title: {unchanged_title}")
    # drop title from praise dicts
    praise["slides"][0]["shapes"] = [shape for shape in praise["slides"][0]["shapes"] if shape.get("text") != unchanged_title]
    praise["title"] = title

In [ ]:
all_praises[2]

## Identificação de bis

In [ ]:
bag_slides_com_bis = []
bis_tag_regex = r"^<bis\b[^>]*>\s*(\n|$)"
for praise in all_praises:
    for slide in praise.get("slides", []):
        for shape in slide.get("shapes", []):
            if "text_processado" in shape and re.search(bis_tag_regex, shape["text_processado"]):
                bag_slides_com_bis.append(slide.copy())
                
len(bag_slides_com_bis)

In [ ]:
bag_slides_com_bis[0]

In [ ]:
for idx, slide in enumerate(bag_slides_com_bis):
    count_bis = sum(1 for shape in slide.get("shapes", []) if "text_processado" in shape and re.search(bis_tag_regex, shape["text_processado"]))
    if count_bis > 1:
        print(f"Slide {idx} ({slide['slide']}) has {count_bis} bis tags.")

In [ ]:
slide_teste = bag_slides_com_bis[0]
textos_processados = ""
textos = ""
height_texto = 0
top_texto = 0
height_brace = []
top_brace = []

for shape in slide_teste["shapes"]:
    if (shape.get("shape") == "AUTO_SHAPE" and shape.get("auto_shape") == "RIGHT_BRACE") or shape.get("shape") == "FREEFORM":
        height_brace.append(shape.get("height", 0))
        top_brace.append(shape.get("top", 0))

    if "text_processado" in shape and not re.search(bis_tag_regex, shape["text_processado"]):
        height_texto += shape.get("height", 0)
        top_texto += shape.get("top", 0)
        textos_processados += shape["text_processado"]
        textos += shape.get("text", "")

print(textos_processados)
print(textos)
height_brace, top_brace, height_texto, top_texto

In [ ]:
import math
qtde_linhas = len(textos_processados.split("\n"))
height_linha = height_texto / qtde_linhas if qtde_linhas > 0 else 0
qtde_linhas_brace = [math.ceil(h / height_linha if height_linha > 0 else 0) for h in height_brace]
qtde_linhas_brace

In [ ]:
linha_inicio_brace = [int(round(((top - top_texto) / height_linha if height_linha > 0 else 0), 0)) for top in top_brace]
linha_inicio_brace

In [ ]:
linhas = textos_processados.split("\n")
print(linhas)

for linha_inicio, qtde_linhas in zip(linha_inicio_brace, qtde_linhas_brace):
    for idx, line in enumerate(linhas):
        if idx >= linha_inicio and qtde_linhas > 0:
            linhas[idx] = line + "<bis>"
            qtde_linhas -= 1

textos_processados = "\n".join(linhas)
print(textos_processados)

In [ ]:
new_shape = {
    "shape": "AUTO_SHAPE",
    "auto_shape": "RECTANGLE",
    "text_processado": textos_processados,
    "text": textos
}
slide_teste["shapes"] = [new_shape]
slide_teste

### Para todos

In [ ]:
import math
import re

# bis_tag_regex = r"^<bis\b[^>]*>\s*(\n|$)"
bis_tag_regex = r"^<bis(\s+modo=\".*\")?>\s*(\n|$)"


def has_bis_tag(shape: dict, pattern: str) -> bool:
    text = shape.get("text_processado", "")
    return bool(text) and re.search(pattern, text) is not None


def build_merged_shape_for_bis(slide: dict, pattern: str) -> dict | None:
    shapes = slide.get("shapes", [])

    # Processa o slide apenas se existir algum shape com tag <bis> no inicio do texto.
    if not any(has_bis_tag(shape, pattern) for shape in shapes):
        return None

    textos_processados = []
    textos = []
    height_texto = 0
    top_texto = 0
    height_brace = []
    top_brace = []
    modo_bis = None
    bis_tag = "<bis>"

    for shape in shapes:
        is_brace = (
            shape.get("shape") == "AUTO_SHAPE"
            and shape.get("auto_shape") == "RIGHT_BRACE"
        ) or shape.get("shape") == "FREEFORM"

        if is_brace:
            height_brace.append(shape.get("height", 0))
            top_brace.append(shape.get("top", 0))

        if "text_processado" in shape and not has_bis_tag(shape, pattern):
            height_texto += shape.get("height", 0)
            top_texto += shape.get("top", 0)
            textos_processados.append(shape["text_processado"])
            textos.append(shape.get("text", ""))

        if has_bis_tag(shape, pattern):
            modo_bis = re.search(pattern, shape["text_processado"], re.IGNORECASE).group(1)
            bis_tag = f"<bis{modo_bis}>" if modo_bis else "<bis>"

    texto_processado_unico = "".join(textos_processados)
    texto_unico = "".join(textos)

    qtde_linhas_texto = len(texto_processado_unico.split("\n"))
    height_linha = height_texto / qtde_linhas_texto if qtde_linhas_texto > 0 else 0

    qtde_linhas_brace = [
        math.ceil(h / height_linha if height_linha > 0 else 0)
        for h in height_brace
    ]
    linha_inicio_brace = [
        int(round(((top - top_texto) / height_linha if height_linha > 0 else 0), 0))
        for top in top_brace
    ]

    linhas = texto_processado_unico.split("\n")

    for linha_inicio, linhas_a_marcar in zip(linha_inicio_brace, qtde_linhas_brace):
        for idx, line in enumerate(linhas):
            if idx >= linha_inicio and linhas_a_marcar > 0:
                if bis_tag not in line:
                    linhas[idx] = line + bis_tag
                linhas_a_marcar -= 1

    texto_processado_unico = "\n".join(linhas)

    return {
        "shape": "AUTO_SHAPE",
        "auto_shape": "RECTANGLE",
        "text_processado": texto_processado_unico,
        "text": texto_unico,
    }


for praise in all_praises:
    for slide in praise.get("slides", []):
        new_shape = build_merged_shape_for_bis(slide, bis_tag_regex)
        if new_shape is not None:
            slide["shapes"] = [new_shape]

In [ ]:
all_praises[2]

## Nova estrutura

In [ ]:
def set_text_full(texts):
    texts_full = [
        line
        for line in texts
        if line.strip() != ""
    ]
    texts_full = "\n\n".join(texts_full)
    texts_full = texts_full.replace('"', "'")
    # standardize multiple whitespaces to single space
    texts_full = re.sub(r"\s+", " ", texts_full).strip()
    return texts_full


In [ ]:
def return_number(title):
    if title:
        match = re.search(r"^\d+", title)
        if match:
            return match.group()
    return "null"


In [ ]:
# TODO:
# - Não precisa preencher a coluna de texto limpo, só faz sentido na hora de NLP
# - Dá pra otimizar os <bis>, com linhas que se repetem

new_structure = []
for praise in all_praises:
    texts_processado = list(get_texts(praise, "text_processado"))
    texts = list(get_texts(praise, "text"))

    if not texts_processado:
        continue

    titulo = praise.get("title", "null")
    numero_regex = r"^(\d+)\s+-\s+"
    if re.match(numero_regex, titulo):
        numero = re.match(numero_regex, titulo).group(1)
        titulo = titulo[len(numero)+3:].strip()
    else:        
        numero = "null"

    texts_processado_full = set_text_full(texts_processado)
    texts_full = set_text_full(texts)

    new_structure.append(
        {
            "numero": numero,
            "nome": titulo if titulo is not None else "null",
            "texto_processado": texts_processado_full,
            "texto": texts_full,
        }
    )

print(new_structure[2]["texto_processado"])
new_structure[2]

In [ ]:
new_structure

# Criar arquivo de inserção

In [ ]:
import glob

files_json = glob.glob("slides_json\\*.json")
files_json

In [ ]:
from pathlib import Path

for index, file in enumerate(files_json):
    file_name = (
        "00"
        + str(index + 3)
        + "-add-"
        + Path(file).stem.lower().replace(" ", "-")
        + ".sql"
    )

    with open(file, "r") as f:
        louvores = json.load(f)

    with open(str(Path("..//..//database//migrations") / file_name), "w") as f:
        for hino in louvores:
            text = (
                "INSERT INTO hino (numero, nome, nome_pt, texto, texto_processado, coletanea_id, idioma, date_insert, date_update) VALUES ('"
                + str(int(hino["numero"]) if hino["numero"].isdigit() else "null")
                + "', '"
                + hino["nome"]
                + "', '"
                + hino["nome"]
                + "', '"
                + hino["texto"].replace("\n", "\\n").replace("'", "''")
                + "', '"
                + hino["texto_processado"].replace("\n", "\\n").replace("'", "''")
                + "', "
                + str(index + 1)
                + ", PT-BR, CURRENT_TIMESTAMP, CURRENT_TIMESTAMP);\n"
            )
            f.write(text)